In [2]:
# Cài thư viện
import osmnx as ox #open street map: mô hình hóa và trực quan hóa mạng lưới giao thông
import requests # gọi api
import pandas as pd # Xử lý phân tích và làm sạch dữ liệu
import numpy as np # tính toán
import time # thời gian
from datetime import datetime, timedelta
import networkx as nx # thư viện dùng để tạo, nghiên cứu cấu trúc của grap
import folium
from datetime import datetime, timedelta

In [3]:
REAL_DATA_FILE = 'hanoi_traffic_FULL_SCAN_20251127_1504.csv'

In [4]:
CENTER_POINT = (21.0025, 105.8037) 
# Bán kính 3km (đủ bao trùm Vành đai 3 và các đường lân cận như Nguyễn Trãi, Khuất Duy Tiến)
DISTANCE_M = 3000
# Bỏ qua đường dân sinh, đường nội bộ chỉ lấy cao tốc, quốc lộ, đường trục, đườn cấp 1, cấp 2
cf = '["highway"~"motorway|trunk|primary|secondary"]'
# Cấu hình thời gian (7 ngày, 5 phút/lần)
FREQ_MIN = 5
TOTAL_STEPS = (24 * 60 // FREQ_MIN) * DAYS # 2016 bước thời gian
MAX_EDGES_TO_SAMPLE = 1000 # giới hạn 1000 cạnh
print(f"🚀 Đang khởi tạo quy trình sinh dữ liệu cho {DAYS} ngày ({TOTAL_STEPS} bước)...")

🚀 Đang khởi tạo quy trình sinh dữ liệu cho 1 ngày (288 bước)...


In [5]:
G = ox.graph_from_point(
    CENTER_POINT, 
    dist=DISTANCE_M, 
    network_type="drive",
    custom_filter=cf
) # lấy grap
# Lấy danh sách các Nút (Nodes) - Đây chính là N trong mô hình T-GCN
nodes_gdf = ox.graph_to_gdfs(G, edges=False, nodes=True)
node_ids = nodes_gdf.index.tolist()
num_nodes = len(node_ids)

# Tạo Ma trận Kề (Adjacency Matrix)
adj_matrix = nx.adjacency_matrix(G).todense()

print(f"✅ Đã tạo Graph thành công!")
print(f"   - Số lượng Nút (Nodes/Sensors): {num_nodes}")
print(f"   - Kích thước Ma trận kề: {adj_matrix.shape}")

✅ Đã tạo Graph thành công!
   - Số lượng Nút (Nodes/Sensors): 624
   - Kích thước Ma trận kề: (624, 624)


In [7]:
try:
    df_real = pd.read_csv(REAL_DATA_FILE)
    REAL_MEAN_SPEED = df_real['speed_kmh'].mean()
    REAL_STD_DEV = df_real['speed_kmh'].std()
    
    # Lấy danh sách các Edge IDs thực tế mà ta đã lấy mẫu
    real_edge_ids = df_real['edge_id'].unique()
    
    # Tính tốc độ thông thoáng lý tưởng (Free Flow Speed)
    # Giả định Tốc độ lý tưởng cao hơn tốc độ trung bình thực tế
    FREE_FLOW_SPEED = REAL_MEAN_SPEED + 1.5 * REAL_STD_DEV
    
    print(f"✅ Calibration thành công! Trung bình thực tế: {REAL_MEAN_SPEED:.2f} km/h")
    print(f"   - Tốc độ Free Flow được thiết lập: {FREE_FLOW_SPEED:.2f} km/h")

except FileNotFoundError:
    print(f"🚨 LỖI: Không tìm thấy file {REAL_DATA_FILE}. Sử dụng giá trị mặc định.")
    FREE_FLOW_SPEED = 35.0
    REAL_MEAN_SPEED = 25.0
    real_edge_ids = [] # Dùng danh sách cạnh của G nếu không có file

✅ Calibration thành công! Trung bình thực tế: 15.51 km/h
   - Tốc độ Free Flow được thiết lập: 29.29 km/h


In [10]:
DAYS = 365
FREQ_MIN = 5
TOTAL_STEPS = (24 * 60 // FREQ_MIN) * DAYS


# Tỷ lệ ảnh hưởng cho mỗi yếu tố (W < 1 là giảm tốc độ)
W_RAIN = 0.70 # Giảm 30% tốc độ khi mưa 
W_FLOOD = 0.40 # Giảm 60% khi lụt
W_ACCIDENT = 0.60 # Giảm 60% tốc độ tại điểm tai nạn
W_MONDAY_AM = 0.90 # Tăng thêm 15% tắc nghẽn sáng T2
W_FRIDAY_PM = 0.80 # Tăng thêm 10% tắc nghẽn chiều T6

# Danh sách các ngày lễ quan trọng để áp dụng trọng số đặc biệt
holidays_list = [
    datetime(2024, 1, 1).date(), # Tết Dương Lịch
    datetime(2024, 4, 30).date(), # Giải phóng miền Nam
    datetime(2024, 5, 1).date(), # Quốc tế Lao động
    datetime(2024, 9, 2).date(), # Quốc khánh
]

event_list = [
    datetime(2024, 3, 8).date(), 
    datetime(2024, 6, 1).date(), 
    datetime(2024, 12, 24).date(),
    datetime(2024, 2, 14).date(),
    datetime(2024, 10, 31).date(),
    datetime(2024,10,20).date()
]

SCHOOL_BREAK_START = datetime(2025, 6, 1).date() # HS nghỉ hè
SCHOOL_BREAK_END = datetime(2025, 8, 15).date() # Kết thức nghỉ hè

WINTER = datetime(2024, 12, 1).date()
AUTUMN = datetime(2024, 9, 1).date()
AUTUMN = datetime(2024, 5, 1).date()
SPRING = datetime(2024, 2, 1).date()

COUNT_ACCIDENT = 250 # Vụ tai nạn giao thông ở Hà Nội năm 2024 vì tổng là có 1500 vụ (chiến khoảng 16%)

In [ ]:
def generate_synthetic_data(n_steps, n_nodes):
    time_index = pd.date_range(start=datetime(2024, 1, 1), periods=n_steps, freq=f'{FREQ_MIN}min') # Trục thời gian
    data_matrix = np.full((n_steps, n_nodes), FREE_FLOW_SPEED) # Ma trận tốc độ nền tảng
    
    # Ma trận Lưu lượng (Flow) và Thời tiết
    flow_matrix = np.zeros_like(data_matrix) # Khởi tạo ma trận có kích thước = data_matrix với tất cả giá trị 0
    weather_condition = np.zeros(n_steps) # 0=Clear, 1=Rain, 2=Accident
    temperature = np.zeros(n_steps)

    # Lấy mẫu các nút giao thông lớn (đường chính) để tạo sự khác biệt
    # Ví dụ: 5% các nút là nút giao lớn (Nguyễn Trãi, Trường Chinh)
    major_node_indices = np.random.choice(n_nodes, size=int(n_nodes * 0.05), replace=False)
    
    # Biến trạng thái Tai nạn (Accident)
    accident_state = {'active': False, 'node': None, 'end_step': 0}
    
    for t in range(n_steps):
        dt = time_index[t]
        hour = dt.hour
        weekday = dt.weekday() # 0=Thứ Hai, 6=Chủ Nhật
        
        # 1. TÍNH TOÁN TRỌNG SỐ THỜI GIAN (W_Daily & W_DoW)
        W_time = 1.0 
        
        # Giờ cao điểm (7:00-8:30)
        if (7 <= hour <= 8 and (hour < 8 or dt.minute <= 30)): 
            W_time = np.random.uniform(0.35, 0.5) 
            if weekday == 0: W_time *= W_MONDAY_AM # Sáng T2 tệ hơn
        
        # Giờ cao điểm (16:00-18:30)
        elif (16 <= hour <= 18 and (hour < 18 or dt.minute <= 30)):
            W_time = np.random.uniform(0.4, 0.6)
            if weekday == 4: W_time *= W_FRIDAY_PM # Chiều T6 tệ hơn
        
        # 2. TÍNH TOÁN TRỌNG SỐ NGOÀI (W_External: W_Weather & W_Holiday)
        
        # A. Ảnh hưởng Ngày Lễ
        if dt.date() in holidays_list: 
            W_time *= 0.5 # Giảm tắc nghẽn (Đường vắng)
        
        # B. Ảnh hưởng Nghỉ hè
        if SCHOOL_BREAK_START <= dt.date() <= SCHOOL_BREAK_END:
            W_time *= 1.1 # Tốc độ tăng 10%
            
        # C. Mô phỏng Thời tiết (Đơn giản: 15% khả năng mưa bất kỳ lúc nào)
        is_raining = np.random.rand() < 0.15 # 15% khả năng mưa
        if is_raining:
            W_time *= W_RAIN
            weather_condition[t] = 1
        
        # D. Mô phỏng Nhiệt độ (Cyclical + Random)
        day_of_year = dt.timetuple().tm_yday
        # Chu kỳ năm (Hè nóng hơn Đông)
        W_season = 1 + 0.15 * np.sin(2 * np.pi * day_of_year / 365) 
        temperature[t] = 25 + 5 * np.sin(2 * np.pi * hour / 24) * W_season + np.random.normal(0, 1)

        # E. Mô phỏng Tai nạn (Accident)
        W_accident = np.ones(n_nodes)
        
        if not accident_state['active']:
            if np.random.rand() < 0.005: # 0.5% xác suất tai nạn bắt đầu
                node_idx = np.random.choice(major_node_indices) # Tai nạn xảy ra trên đường lớn
                accident_state['active'] = True
                accident_state['node'] = node_idx
                accident_state['end_step'] = t + np.random.randint(6, 12) # Kéo dài 30-60 phút
        
        if accident_state['active']:
            weather_condition[t] = 2 # Ký hiệu là Sự kiện (Accident)
            
            # Áp dụng W_ACCIDENT cho nút bị nạn và các nút lân cận
            node_idx = accident_state['node']
            
            # Giảm tốc độ chính cho nút bị nạn
            W_accident[node_idx] = W_ACCIDENT
            
            # Giảm tốc độ cho các nút lân cận (ảnh hưởng dây chuyền)
            for neighbor_idx in np.where(adj_matrix[node_idx])[1]:
                 W_accident[neighbor_idx] *= 0.8
                 
            # Kiểm tra xem tai nạn đã hết chưa
            if t >= accident_state['end_step']:
                accident_state['active'] = False

        # 3. ÁP DỤNG TRỌNG SỐ CUỐI CÙNG
        for i in range(n_nodes):
            # Tốc độ cuối cùng = Tốc độ lý tưởng * W_Thời gian * W_Tai nạn + Nhiễu
            
            # Logic: Nút giao lớn bị ảnh hưởng bởi tắc nghẽn nhiều hơn (20%)
            W_location = 0.8 if i in major_node_indices else 1.0 
            
            final_weight = W_time * W_accident[i] * W_location
            
            data_matrix[t, i] = FREE_FLOW_SPEED * final_weight + np.random.normal(0, REAL_STD_DEV * 0.1)
            
            # Sinh Lưu lượng (Flow) ngược chiều với Tốc độ
            # Lưu lượng = Base_Flow + (Tắc nghẽn * 100) + Noise
            flow_matrix[t, i] = 50 + (1 - final_weight) * 200 + np.random.normal(0, 10)
        
    # Đảm bảo tốc độ và lưu lượng không âm
    data_matrix = np.clip(data_matrix, 5, FREE_FLOW_SPEED * 1.2)
    flow_matrix = np.clip(flow_matrix, 5, 800)
    
    return data_matrix, flow_matrix, temperature, weather_condition

In [ ]:
print("\n🚀 Bắt đầu sinh dữ liệu 1 năm (105,120 bước thời gian)...")
speed_matrix, flow_matrix, temp_array, condition_array = generate_synthetic_data(TOTAL_STEPS, num_nodes)

# TẠO DATAFRAME VÀ LƯU
df_speed = pd.DataFrame(speed_matrix, index=time_index, columns=node_ids)
df_flow = pd.DataFrame(flow_matrix, index=time_index, columns=node_ids)

# Dữ liệu ngoài (External Data)
df_weather = pd.DataFrame({
    'Temperature_C': temp_array,
    'Condition': condition_array
}, index=time_index)

# Lưu Ma trận Kề (Spatial)
np.save("hanoi_adj_matrix_1year.npy", adj_matrix)
print("   - Ma trận kề đã lưu: hanoi_adj_matrix_1year.npy")

# Lưu các Features (Temporal & External)
df_speed.to_csv("hanoi_speed_1year.csv")
df_flow.to_csv("hanoi_flow_1year.csv")
df_weather.to_csv("hanoi_weather_1year.csv")

print("\n🎉 HOÀN TẤT TẠO BỘ DỮ LIỆU GIẢ LẬP 1 NĂM!")
print(f"   - Tốc độ đã lưu: hanoi_speed_1year.csv ({df_speed.shape} - 52 triệu điểm dữ liệu)")
print("   - Bạn đã có tất cả các thành phần cần thiết cho mô hình T-GCN/STEP.")